# Run the pretrained transmon VQE on IBM Quantum — Open Plan

This notebook is designed for the free IBM Quantum Open Plan. It uses the 25 noiselessly pretrained angles from Step 1 and limits QPU use to two jobs:

1. **Exploration:** the pretrained point plus four SPSA $+/-$ perturbation pairs (9 parameter vectors, 1,024 shots).
2. **Validation:** the pretrained point, one locally calculated SPSA update, and the best exploratory point (3 vectors, 4,096 shots).

The notebook defaults to `RUN_HARDWARE = False`. Run all local validation cells first. Only change the switch after checking the backend, transpiled depth, two-qubit gate count, and your remaining IBM allocation.

Important: the Step 1 simulator numbered wire 0 as the most-significant bit, while Qiskit numbers qubit 0 as the least-significant bit. This notebook reverses the wire mapping and validates the resulting energy before allowing submission.


In [ ]:
# Recommended environment:
# %pip install --upgrade "qiskit~=2.5" "qiskit-ibm-runtime>=0.50" "qiskit-algorithms>=0.4"

import json
from pathlib import Path
import numpy as np

from scipy.constants import e as electron_charge, h as planck_h
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Operator, SparsePauliOp, Statevector
from qiskit.transpiler import generate_preset_pass_manager

RUN_HARDWARE = False       # Change to True only after local validation.
EXPLORATION_SHOTS = 1024
VALIDATION_SHOTS = 4096
RNG_SEED = 42
PARAMETER_FILE = next(
    (p for p in [
        Path("step1_noiseless_vqe_parameters.json"),
        Path("outputs/step1_noiseless_vqe_parameters.json"),
    ] if p.exists()),
    Path("step1_noiseless_vqe_parameters.json"),
)

print("Hardware submission enabled:", RUN_HARDWARE)


## 1. Load one pretrained parameter vector

Keep `step1_noiseless_vqe_parameters.json` in the same directory as this notebook. Seed 11 is used by default; all three exported seeds passed the noiseless readiness gates.


In [ ]:
if not PARAMETER_FILE.exists():
    raise FileNotFoundError(
        f"Cannot find {PARAMETER_FILE.resolve()}. Copy the exported Step 1 JSON beside this notebook."
    )

with PARAMETER_FILE.open() as f:
    saved = json.load(f)

seed_key = "11"
reps = int(saved["selected_reps"])
theta_initial = np.asarray(saved["parameters"][seed_key], dtype=float)
num_qubits = int(saved["model"]["qubits"])

assert reps == 4
assert num_qubits == 5
assert theta_initial.size == num_qubits * (reps + 1) == 25

print(f"Loaded seed {seed_key}: {theta_initial.size} angles, {reps} repetitions")


## 2. Reconstruct the identical circuit and Hamiltonian

The reversal `qiskit_qubit = num_qubits - 1 - custom_wire` is intentional. Removing it changes the state associated with the saved angles.


In [ ]:
theta = ParameterVector("theta", theta_initial.size)
ansatz = QuantumCircuit(num_qubits)
k = 0

for layer in range(reps + 1):
    for custom_wire in range(num_qubits):
        qiskit_qubit = num_qubits - 1 - custom_wire
        ansatz.ry(theta[k], qiskit_qubit)
        k += 1

    if layer < reps:
        for custom_control in range(num_qubits):
            custom_target = (custom_control + 1) % num_qubits
            qiskit_control = num_qubits - 1 - custom_control
            qiskit_target = num_qubits - 1 - custom_target
            ansatz.cx(qiskit_control, qiskit_target)

# Reconstruct the original physical model.
L_J_nH, f_linear_GHz, n_g, N = 10.0, 5.21015, 0.0, 8
L_J = L_J_nH * 1e-9
C_eff = 1 / (L_J * (2*np.pi*f_linear_GHz*1e9)**2)
E_C = electron_charge**2 / (2*C_eff*planck_h) / 1e9
E_J = (planck_h/(4*np.pi*electron_charge))**2 / (L_J*planck_h) / 1e9
n = np.arange(-N, N+1); d = len(n); D = 2**num_qubits
H = (np.diag(4*E_C*(n-n_g)**2)
     + np.diag(np.full(d-1, -E_J/2), 1)
     + np.diag(np.full(d-1, -E_J/2), -1))
levels, vectors = np.linalg.eigh(H)
E0_exact = float(levels[0])
E_pad = float(4*E_C*N*N + E_J)
A = E_pad*np.eye(D); A[:d, :d] = H
hamiltonian = SparsePauliOp.from_operator(Operator(A), atol=1e-12, rtol=0)

print("Logical circuit depth:", ansatz.depth())
print("Logical operations:", dict(ansatz.count_ops()))
print("Pauli terms:", len(hamiltonian))


## 3. Mandatory local validation

Do not submit anything unless this cell reports an error below 5 MHz. It checks the parameter order, wire convention, Hamiltonian, and saved file together.


In [ ]:
bound = ansatz.assign_parameters({theta[i]: theta_initial[i] for i in range(len(theta))})
psi = Statevector.from_instruction(bound).data
local_energy = float(np.vdot(psi, A @ psi).real)
local_error_mhz = 1000 * abs(local_energy - E0_exact)
local_fidelity = float(abs(np.vdot(vectors[:, 0], psi[:d]))**2)
padding_probability = float(np.sum(abs(psi[d:])**2))

print(f"Exact E0:            {E0_exact:.12f} GHz")
print(f"Saved-point energy:  {local_energy:.12f} GHz")
print(f"Error:               {local_error_mhz:.9f} MHz")
print(f"Ground fidelity:     {local_fidelity:.12f}")
print(f"Padding probability: {padding_probability:.3e}")

assert local_error_mhz < 5.0, "STOP: circuit/parameter mapping does not reproduce Step 1."
assert local_fidelity > 0.99
assert padding_probability < 1e-3
print("LOCAL VALIDATION PASSED")


## 4. Connect, select a backend, and transpile

First save your account once in a private cell (never share the token):

```python
from qiskit_ibm_runtime import QiskitRuntimeService
QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token="YOUR_TOKEN", overwrite=True)
```

Open Plan workloads use job or batch mode, not sessions. This notebook uses job mode and puts multiple parameter vectors in each request.


In [ ]:
if RUN_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService

    service = QiskitRuntimeService()
    backend = service.least_busy(
        operational=True,
        simulator=False,
        min_num_qubits=num_qubits,
    )
    print("Selected backend:", backend.name)

    # Try several routing seeds and keep the lowest two-qubit count, then depth.
    transpiled_candidates = []
    for transpiler_seed in range(10):
        pm = generate_preset_pass_manager(
            backend=backend,
            optimization_level=3,
            seed_transpiler=transpiler_seed,
        )
        candidate = pm.run(ansatz)
        twoq = sum(1 for inst in candidate.data if inst.operation.num_qubits == 2)
        transpiled_candidates.append((twoq, candidate.depth(), transpiler_seed, candidate))

    isa_twoq, isa_depth, best_transpiler_seed, isa_ansatz = min(transpiled_candidates, key=lambda x: x[:3])
    hamiltonian_isa = hamiltonian.apply_layout(isa_ansatz.layout)

    # Confirm transpilation preserved the exact ParameterVector identities/order.
    isa_parameters = list(isa_ansatz.parameters)
    assert [str(p) for p in isa_parameters] == [str(p) for p in theta]

    print("Transpiler seed:", best_transpiler_seed)
    print("ISA depth:", isa_depth)
    print("ISA two-qubit gates:", isa_twoq)
    print("ISA operations:", dict(isa_ansatz.count_ops()))
else:
    print("Skipped IBM connection and transpilation. Set RUN_HARDWARE=True when ready.")


## 5. Exploration job: nine parameter vectors

This single job evaluates the pretrained point and four symmetric perturbation pairs. Measurement mitigation is enabled through resilience level 1. ZNE and PEC are intentionally disabled to protect the free allowance.


In [ ]:
if RUN_HARDWARE:
    # Current client-side primitive (qiskit-ibm-runtime >= 0.50), with legacy fallback.
    try:
        from qiskit_ibm_runtime.executor_estimator import Estimator
    except ImportError:
        from qiskit_ibm_runtime import EstimatorV2 as Estimator

    estimator = Estimator(mode=backend)
    estimator.options.default_shots = EXPLORATION_SHOTS
    estimator.options.resilience_level = 1
    estimator.options.resilience.measure_mitigation = True
    estimator.options.resilience.zne_mitigation = False
    estimator.options.resilience.pec_mitigation = False

    rng = np.random.default_rng(RNG_SEED)
    num_directions = 4
    perturbation = 0.08
    directions = rng.choice([-1.0, 1.0], size=(num_directions, theta_initial.size))

    exploration_candidates = [theta_initial]
    for direction in directions:
        exploration_candidates.extend([
            theta_initial + perturbation*direction,
            theta_initial - perturbation*direction,
        ])
    exploration_candidates = (np.asarray(exploration_candidates) + np.pi) % (2*np.pi) - np.pi

    print("About to submit", len(exploration_candidates), "parameter vectors at", EXPLORATION_SHOTS, "shots.")
    exploration_job = estimator.run([
        (isa_ansatz, hamiltonian_isa, exploration_candidates)
    ])
    print("Exploration job ID:", exploration_job.job_id())

    exploration_pub = exploration_job.result()[0]
    exploration_energies = np.asarray(exploration_pub.data.evs, dtype=float).reshape(-1)
    exploration_stds = np.asarray(exploration_pub.data.stds, dtype=float).reshape(-1)

    for i, (energy, std) in enumerate(zip(exploration_energies, exploration_stds)):
        print(f"{i:2d}: E={energy:.9f} GHz, std={std:.6f} GHz")
else:
    print("Exploration job not submitted.")


## 6. Calculate one SPSA update locally

No QPU work occurs in this cell. The four symmetric pairs provide four stochastic gradient estimates, which are averaged and norm-clipped.


In [ ]:
if RUN_HARDWARE:
    gradient_estimates = []
    for k, direction in enumerate(directions):
        e_plus = exploration_energies[1 + 2*k]
        e_minus = exploration_energies[2 + 2*k]
        gradient_estimates.append(((e_plus-e_minus)/(2*perturbation))*direction)

    gradient = np.mean(gradient_estimates, axis=0)
    learning_rate = 0.03
    update = -learning_rate * gradient
    max_update_norm = 0.25
    if np.linalg.norm(update) > max_update_norm:
        update *= max_update_norm / np.linalg.norm(update)

    theta_updated = (theta_initial + update + np.pi) % (2*np.pi) - np.pi
    best_exploration_index = int(np.argmin(exploration_energies))
    theta_best_exploration = exploration_candidates[best_exploration_index]

    print("Gradient norm:", np.linalg.norm(gradient))
    print("Applied update norm:", np.linalg.norm(update))
    print("Best exploratory index:", best_exploration_index)
else:
    print("No hardware exploration results available.")


## 7. Validation job: three candidates

The second and final planned QPU job remeasures the initial point, calculated SPSA update, and lowest exploratory observation with more shots. The lowest exploratory observation is not trusted without this independent validation.


In [ ]:
if RUN_HARDWARE:
    validation_candidates = np.asarray([
        theta_initial,
        theta_updated,
        theta_best_exploration,
    ])
    labels = ["pretrained", "SPSA update", "exploration winner"]

    estimator.options.default_shots = VALIDATION_SHOTS
    validation_job = estimator.run([
        (isa_ansatz, hamiltonian_isa, validation_candidates)
    ])
    print("Validation job ID:", validation_job.job_id())

    validation_pub = validation_job.result()[0]
    validation_energies = np.asarray(validation_pub.data.evs, dtype=float).reshape(-1)
    validation_stds = np.asarray(validation_pub.data.stds, dtype=float).reshape(-1)

    for label, energy, std in zip(labels, validation_energies, validation_stds):
        print(f"{label:20s}: E={energy:.9f} GHz, std={std:.6f} GHz")

    best_index = int(np.argmin(validation_energies))
    improvement = validation_energies[0] - validation_energies[best_index]
    combined_std = float(np.hypot(validation_stds[0], validation_stds[best_index]))

    # Accept only a statistically convincing improvement; otherwise keep the pretrained point.
    accepted_index = best_index if improvement > 2*combined_std else 0
    theta_hardware_best = validation_candidates[accepted_index]
    print("Measured improvement:", improvement, "GHz")
    print("Combined uncertainty:", combined_std, "GHz")
    print("Accepted:", labels[accepted_index])
else:
    print("Validation job not submitted.")


## 8. Save all job IDs, measurements, and selected angles

The complete history is saved so the experiment remains reproducible. Keep the noiseless pretrained vector if the hardware improvement is not at least twice the combined reported uncertainty.


In [ ]:
if RUN_HARDWARE:
    output = {
        "backend": backend.name,
        "transpiler_seed": best_transpiler_seed,
        "isa_depth": isa_depth,
        "isa_two_qubit_gates": isa_twoq,
        "exploration_job_id": exploration_job.job_id(),
        "validation_job_id": validation_job.job_id(),
        "exploration_shots": EXPLORATION_SHOTS,
        "validation_shots": VALIDATION_SHOTS,
        "exploration_energies_GHz": exploration_energies.tolist(),
        "exploration_stds_GHz": exploration_stds.tolist(),
        "validation_labels": labels,
        "validation_energies_GHz": validation_energies.tolist(),
        "validation_stds_GHz": validation_stds.tolist(),
        "accepted_label": labels[accepted_index],
        "selected_parameters": theta_hardware_best.tolist(),
    }
    with open("ibm_open_plan_transmon_vqe_result.json", "w") as f:
        json.dump(output, f, indent=2)
    print("Wrote ibm_open_plan_transmon_vqe_result.json")
else:
    print("Nothing saved because no QPU jobs were submitted.")
